# DL4ES — 2025/2026  
## Домашнее задание: классификация эпизодов загрязнения воздуха (Beijing AQ & Weather)  

---

## Описание датасета (ключевое)

Коллекция данных **Beijing Air Quality & Weather**: почасовые наблюдения (UTC+8) качества воздуха и метеопараметров в Пекине за 2010–2014 годы. Данные могут быть представлены отдельными CSV по станциям. Есть пропуски, сезонность, возможны экстремальные эпизоды смога. fileciteturn4file0

**Используемые колонки:**

- `pm2.5` — концентрация PM2.5 (µg/m³) — из неё строим целевую переменную;
- `DEWP`, `TEMP`, `PRES` — метеопараметры (°C, °C, hPa);
- `Iws` — средняя скорость ветра (m/s);
- `Is`, `Ir` — накопленные часы снега/дождя (hr);
- `cbwd` — направление ветра (категориальная) — **опционально** (бонус).


## 0. Скачивание данных (обязательно сделать «ручками»)

Данные нужно скачать через браузер и положить в локальную папку проекта.

### Шаги

1. Откройте ссылку на архив с данными:  
   **https://ml4es.ru/links/uci-baq**

2. Скачайте архив (обычно это `.zip`).

3. Распакуйте архив так, чтобы в вашем проекте получилась структура (пример):

```
./
|-your_notebook.ipynb
|-data/
|------uci-baq/
|--------------(один или несколько .csv файлов)

```

- Папка `data/uci-baq/` должна содержать CSV-файлы с данными.
- Имена файлов могут отличаться — код ниже ищет CSV автоматически.

4. Запустите ячейку проверки: она покажет, что файлы на месте.

> Если у вас другая структура — **измените путь** `DATA_DIR` в разделе 2.


## 0.1. Метаданные коллекции данных

Ниже — краткая выжимка (полное описание — в `README.md` в раздаче).

- Формат: CSV, разделитель `,`, десятичный `.`  
- Дискретность: 1 час (каждая строка — одно часовое наблюдение)  
- Период: 2010–2014  
- Есть пропуски (`NA`), выраженная сезонность, возможны экстремальные пики `pm2.5`  
- Колонки времени: `year`, `month`, `day`, `hour` (UTC+8)  
- Основные признаки: `DEWP`, `TEMP`, `PRES`, `cbwd`, `Iws`, `Is`, `Ir`  
- Целевая переменная в этом ДЗ строится из `pm2.5`.


## 1. Импорт библиотек

In [1]:
import glob
import pathlib

import numpy as np
import pandas as pd

## 2. Параметры запуска

In [ ]:
# Папка с CSV-файлами датасета (см. инструкцию в разделе 0)
DATA_DIR = pathlib.Path("./data/uci-baq")

# Куда складывать результаты/картинки (если нужно)
OUT_DIR = pathlib.Path("./results_hw_beijing_aq")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR :", OUT_DIR.resolve())


In [ ]:
# Проверка наличия данных
csv_files = sorted(glob.glob(str(DATA_DIR / "**" / "*.csv"), recursive=True))
print(f"Найдено CSV-файлов: {len(csv_files)}")
if len(csv_files) == 0:
    raise FileNotFoundError(
        "Не найдено ни одного CSV. Проверьте, что вы распаковали архив в ./data/uci-baq/ "
        "или поправьте переменную DATA_DIR."
    )
print("Примеры файлов:")
for f in csv_files[:10]:
    print(" -", f)


## 3. Чтение данных

In [ ]:
# Чтение и объединение CSV (если их несколько)
dfs = []
for f in csv_files:
    df_part = pd.read_csv(f)
    dfs.append(df_part)

df = pd.concat(dfs, axis=0, ignore_index=True)
print("Shape:", df.shape)
df.head()


In [ ]:
# Базовая проверка колонок
required_cols = ["year", "month", "day", "hour", "pm2.5", "DEWP", "TEMP", "PRES", "Iws", "Is", "Ir"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        "В данных отсутствуют обязательные колонки: "
        f"{missing}\n"
        "Проверьте, что вы скачали правильный архив (по ссылке из ДЗ) и что читаете правильные CSV."
    )

if "cbwd" not in df.columns:
    print("Колонка 'cbwd' не найдена — это ок, бонус-часть про категориальный признак пропускайте.")

print("OK: обязательные колонки присутствуют.")


In [ ]:
# Соберём datetime (UTC+8, без преобразования TZ)
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]], errors="coerce")
df = df.sort_values("datetime").reset_index(drop=True)
df[["datetime", "pm2.5", "TEMP", "PRES"]].head()


## 4. Разведочный анализ (EDA)

In [ ]:
display(df.describe(include="all"))
print("\nNaN по колонкам (топ-15):")
display(df.isna().mean().sort_values(ascending=False).head(15))


In [ ]:
# Распределение pm2.5 (сырые данные)
pm = df["pm2.5"].dropna().values
plt.figure()
plt.hist(pm, bins=80)
plt.title("pm2.5 distribution (raw)")
plt.xlabel("pm2.5, µg/m³")
plt.ylabel("count")
plt.show()


In [ ]:
# (опционально) сезонность/динамика: pm2.5 vs время (подвыборка для скорости)
df_small = df.dropna(subset=["pm2.5", "datetime"]).iloc[::50].copy()
plt.figure()
plt.plot(df_small["datetime"].values, df_small["pm2.5"].values)
plt.title("pm2.5 time series (subsample)")
plt.xlabel("time")
plt.ylabel("pm2.5, µg/m³")
plt.show()


## 5. Постановка задачи классификации

### 5.1. Целевая переменная

Мы решаем бинарную задачу:

- `y = 1`, если **pm2.5 > 75** µg/m³  
- `y = 0`, если **pm2.5 ≤ 75** µg/m³

> Порог 75 выбран как «эпизод повышенного загрязнения» (условный операционный порог для учебной задачи).

**Важно:** в данных есть пропуски. Строки с пропуском в `pm2.5` или в выбранных признаках использовать нельзя.


In [12]:
FEATURES_NUM = ["DEWP", "TEMP", "PRES", "Iws", "Is", "Ir"]
THRESHOLD_PM25 = 75.0

# TODO:
# 1) Удалите строки, где есть NaN хотя бы в одном из признаков FEATURES_NUM или в 'pm2.5' или в 'datetime'.
# 2) Создайте бинарный столбец 'y' по порогу THRESHOLD_PM25.
# 3) Сформируйте X (numpy array shape [N, d]) и y (numpy array shape [N]).
# 4) Выведите:
#    - итоговый N
#    - долю положительного класса (y==1)
raise NotImplementedError("Сформируйте X и y в этой ячейке.")


NotImplementedError: Сформируйте X и y в этой ячейке.

## 6. Разбиение на train/test

### 6.1. Time-based split

Используем **временной split**:

- первые 80% наблюдений по времени → `train`
- последние 20% → `test`


In [13]:
# TODO:
# 1) Разбейте X и y на X_train, X_test, y_train, y_test (80/20 по времени).
# 2) Выведите размеры получившихся массивов.
# 3) Выведите долю положительного класса отдельно на train и на test.
raise NotImplementedError("Сделайте time-based split.")


NotImplementedError: Сделайте time-based split.

## Далее - ваше решение с использованием искусственных нейронных сетей.